In [6]:
# Movie Review Analyzer System 

In [1]:
# Input Text
#     ↓
# BERT Embeddings (learns general language)
#     ↓
# 12 Transformer Layers (learns relationships)
#     ↓
# [CLS] Token Pooling (sentence-level representation)
#     ↓
# New Classification Head (learns sentiment: 3 classes)
#     ↓
# Output: Positive/Negative/Neutral + Confidence Score

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset, Dataset
import numpy as np 
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
import torch 



In [17]:
class sentimentAnalyzer:
    
    def __init__(self, model_name = 'distilbert-base-uncased'):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None 
        self.class_name = {0:"negative",
                           1: "Neutral",
                           2:"Positve"
                           }
        self.class_to_id = {v: k for k, v in self.class_name.items()}

    def create_sample_dataset(self):
        texts = [
            "This movie was absolutely terrible, worst waste of time!",
            "The film had some good moments but mostly boring.",
            "Amazing movie! Outstanding performances and direction!",
            "It's okay, nothing special but watchable.",
            "Incredible cinematography and brilliant acting!",
            "Disappointing and predictable, very bad.",
            "Pretty good, enjoyed most of it.",
            "One of the best films I've ever seen!",
            "Horrible plot, wasted my evening.",
            "Decent movie, nothing exceptional though.",

        ]
        labels = [0, 1, 2, 1, 2, 0, 1, 2, 0, 1]  # 0=Neg, 1=Neutral, 2=Pos

        # create dataset 
        dataset = Dataset.from_dict({
            'text':texts,
            'labels':labels
        })

        # spli 70% rain, 30 % test 

        split_dataset = dataset.train_test_split(test_size=0.3, seed=42)
        return split_dataset



    def preprocess_data(self,dataset):
        # tokenizes all texts in the dataset 
        # process
        # 1. Split text into tokens 
        # 2. Convert tokens to IDs
        # 3. Add padding and attention masks 
        # 4. return tensors ready for model 

        def tokenize_function(examples):
            return self.tokenizer(
                examples['text'],
                padding = 'max_length',
                truncation = True,
                max_length = 512
            )
        
        tokenized_dataset = dataset.map(
            tokenize_function,
            batched = True,
            remove_columns = ['text']
        )
        return tokenized_dataset

    def train(self, dataset = None, num_epochs = 2):

        # fine tunes the model on sentiment data 
        # steps 
        # 1. Preprocess data (tokenize)
        # load base model
        # define trainign args 
        # create trainer 
        # train and evaluate

        if dataset is None:
            dataset = self.create_sample_dataset()

            # preprocess 
            tokenized_dataset = self.preprocess_data(dataset)

            # load model for classification 
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels =  3
            )


            # definign training args 
            training_args = TrainingArguments(
                output_dir = "./sentiment_model",
                learning_rate = 2e-5, 

                per_device_train_batch_size = 8,
                per_device_eval_batch_size = 8,
                num_train_epochs = num_epochs,
                weight_decay = 0.01,
                eval_strategy = 'epoch',

                save_strategy = "epoch",
                load_best_model_at_end = True,
                logging_steps = 5,
                seed = 42
            )



            def compute_metrics(eval_pred):
                logits, labels = eval_pred
                predictions = np.argmax(logits, axis=-1)

                accuracy = accuracy_score(labels, predictions)
                precision, recall, f1, _ = precision_recall_fscore_support(labels,predictions, average="weighted")

                return {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1
                }

            # create trainer 
            trainer = Trainer(
                model = self.model,
                args = training_args,
                train_dataset = tokenized_dataset['train'],
                eval_dataset = tokenized_dataset['test'],
                data_collator = DataCollatorWithPadding(self.tokenizer),
                compute_metrics = compute_metrics,
                tokenizer = self.tokenizer


            )

            # train 
            return trainer 

    def predict(self, texts):
        if self.model is None:
            raise ValueError("Model is not trained")

        results = []
        self.model.eval()

        for text in texts:
            inputs = self.tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits

            probabilities = torch.nn.functional.softmax(logits, dim=-1)
            predicted_class = torch.argmax(probabilities, dim=-1).item()
            confidence = probabilities[0][predicted_class].item()

            scores = {}
            for class_id, class_name in self.class_name.items():
                if class_id < probabilities.shape[1]:
                    scores[class_name] = probabilities[0][class_id].item()

            results.append({
                'text': text,
                'sentiment': self.class_name[predicted_class],
                'confidence': confidence,
                'scores': scores
            })

        return results


In [16]:
from transformers import AutoModelForSequenceClassification

analyzer = sentimentAnalyzer()
analyzer.model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased-finetuned-sst-2-english'
)
analyzer.model.to(analyzer.device)  # make sure device placement is correct
analyzer.class_name = {0: "negative", 1: "positive"}  # Update class names to match the model

reviews = [
    "this movie was amazing",
    "terrible flim, waste of time"
]

results = analyzer.predict(reviews)

for result in results:
    print(f"text: {result['text']}")
    print(f"sentiment: {result['sentiment']} ({result['confidence']:.2%})")


IndexError: index 2 is out of bounds for dimension 0 with size 2